# CEHARPS 03 — ฝึก Random Forest และ XGBoost / Train Random Forest and XGBoost

ใช้ validation RMSE เลือกโมเดล แล้วประเมิน test เพียงครั้งเดียวหลังเลือก เพื่อลดความเสี่ยงจากการปรับโมเดลตามชุดทดสอบ


In [ ]:
import json  # TH: นำเข้าเครื่องมือ JSON | EN: Import JSON utilities.
import subprocess  # TH: นำเข้าเครื่องมือเรียกคำสั่งระบบ | EN: Import subprocess utilities.
import sys  # TH: นำเข้าข้อมูลตัวแปลภาษา Python | EN: Import Python runtime information.
from pathlib import Path  # TH: นำเข้าคลาสจัดการพาธ | EN: Import the path-management class.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn>=1.5,<2", "xgboost>=3.1,<4", "joblib>=1.4,<2"])  # TH: ติดตั้งไลบรารีฝึกโมเดล | EN: Install model-training libraries.
import joblib  # TH: นำเข้าเครื่องมือบันทึกโมเดล | EN: Import model-serialization utilities.
import numpy as np  # TH: นำเข้า NumPy | EN: Import NumPy.
import pandas as pd  # TH: นำเข้า pandas | EN: Import pandas.
from google.colab import drive  # TH: นำเข้าเครื่องมือเชื่อม Drive | EN: Import the Drive connector.
from sklearn.ensemble import RandomForestRegressor  # TH: นำเข้า Random Forest แบบถดถอย | EN: Import the Random Forest regressor.
from sklearn.impute import SimpleImputer  # TH: นำเข้าเครื่องมือเติมค่าที่หาย | EN: Import missing-value imputation.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # TH: นำเข้าตัวชี้วัดการถดถอย | EN: Import regression metrics.
from sklearn.pipeline import Pipeline  # TH: นำเข้าโครงสร้าง pipeline | EN: Import the pipeline structure.
from xgboost import XGBRegressor  # TH: นำเข้า XGBoost แบบถดถอย | EN: Import the XGBoost regressor.
drive.mount("/content/drive")  # TH: เชื่อม Google Drive | EN: Mount Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CEHARPS")  # TH: กำหนดโฟลเดอร์โครงการ | EN: Define the project folder.
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))  # TH: อ่านค่ากลาง | EN: Load shared settings.
META = json.loads((PROJECT_ROOT / "data/processed/health_metadata.json").read_text(encoding="utf-8"))  # TH: อ่านเมทาดาทาข้อมูลฝึก | EN: Load training-data metadata.
SEED = int(CONFIG["seed"])  # TH: อ่านค่าเมล็ดสุ่ม | EN: Read the random seed.
TARGET = str(META["target"])  # TH: อ่านชื่อเป้าหมาย | EN: Read the target name.
FEATURES = list(META["features"])  # TH: อ่านรายชื่อตัวแปรอิสระ | EN: Read the feature list.


In [ ]:
processed = PROJECT_ROOT / "data/processed"  # TH: กำหนดโฟลเดอร์ข้อมูลพร้อมฝึก | EN: Define the processed-data folder.
train = pd.read_csv(processed / "health_train.csv")  # TH: อ่านชุดฝึก | EN: Load the training split.
val = pd.read_csv(processed / "health_val.csv")  # TH: อ่านชุด validation | EN: Load the validation split.
test = pd.read_csv(processed / "health_test.csv")  # TH: อ่านชุดทดสอบ | EN: Load the test split.
if min(len(train), len(val), len(test)) == 0:  # TH: ตรวจว่าทุกชุดมีข้อมูล | EN: Check that every split contains rows.
    raise ValueError("One or more data splits are empty")  # TH: หยุดเมื่อพบชุดว่าง | EN: Stop when any split is empty.
X_train, y_train = train[FEATURES], train[TARGET]  # TH: แยกตัวแปรและเป้าหมายของชุดฝึก | EN: Separate training features and target.
X_val, y_val = val[FEATURES], val[TARGET]  # TH: แยกตัวแปรและเป้าหมายของชุด validation | EN: Separate validation features and target.
X_test, y_test = test[FEATURES], test[TARGET]  # TH: แยกตัวแปรและเป้าหมายของชุดทดสอบ | EN: Separate test features and target.

def metrics(y_true, y_pred) -> dict:  # TH: สร้างฟังก์ชันคำนวณตัวชี้วัด | EN: Define a metric-calculation function.
    return {"MAE": float(mean_absolute_error(y_true, y_pred)), "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5), "R2": float(r2_score(y_true, y_pred))}  # TH: คืนค่า MAE RMSE และ R² | EN: Return MAE, RMSE, and R².

models = {  # TH: เริ่มกำหนดโมเดลที่ต้องเปรียบเทียบ | EN: Start defining models to compare.
    "random_forest": RandomForestRegressor(n_estimators=300, max_features="sqrt", min_samples_leaf=2, n_jobs=-1, random_state=SEED),  # TH: กำหนด Random Forest baseline | EN: Define the Random Forest baseline.
    "xgboost": XGBRegressor(objective="reg:squarederror", n_estimators=500, learning_rate=0.03, max_depth=6, min_child_weight=3, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.05, reg_lambda=1.0, tree_method="hist", device=str(CONFIG["xgb_device"]), n_jobs=-1, random_state=SEED),  # TH: กำหนด XGBoost โมเดลหลัก | EN: Define the main XGBoost model.
}  # TH: ปิดพจนานุกรมโมเดล | EN: Close the model dictionary.


In [ ]:
artifacts = PROJECT_ROOT / "artifacts/health"  # TH: กำหนดโฟลเดอร์ผลโมเดลสุขภาพ | EN: Define the health-model artifact folder.
artifacts.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์ผลลัพธ์ | EN: Create the artifact folder.
fitted = {}  # TH: เตรียมพจนานุกรมเก็บโมเดลที่ฝึกแล้ว | EN: Initialize fitted-model storage.
validation_rows = []  # TH: เตรียมรายการผล validation | EN: Initialize validation-result records.
for name, estimator in models.items():  # TH: วนฝึกโมเดลแต่ละชนิด | EN: Iterate through each model.
    pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])  # TH: เติมค่าหายจากชุดฝึกก่อนเข้าโมเดล | EN: Impute from training data before modeling.
    pipeline.fit(X_train, y_train)  # TH: ฝึก pipeline ด้วยชุดฝึกเท่านั้น | EN: Fit the pipeline on training data only.
    val_prediction = np.clip(pipeline.predict(X_val), 0, 100)  # TH: พยากรณ์ validation และจำกัดช่วง 0–100 | EN: Predict validation values and clip to 0–100.
    row = {"model": name, "split": "val", **metrics(y_val, val_prediction)}  # TH: บันทึกผล validation | EN: Record validation metrics.
    validation_rows.append(row)  # TH: เพิ่มผลลงรายการ | EN: Append the validation result.
    joblib.dump(pipeline, artifacts / f"{TARGET}_{name}.joblib")  # TH: บันทึกโมเดลปัจจุบัน | EN: Save the current model.
    fitted[name] = pipeline  # TH: เก็บโมเดลในหน่วยความจำ | EN: Keep the model in memory.
    print(row)  # TH: แสดงผล validation | EN: Display validation metrics.
validation = pd.DataFrame(validation_rows).sort_values("RMSE")  # TH: เรียงโมเดลจาก RMSE ต่ำสุด | EN: Sort models by lowest RMSE.
selected_name = str(validation.iloc[0]["model"])  # TH: เลือกโมเดลที่ดีที่สุดบน validation | EN: Select the best validation model.
selected = fitted[selected_name]  # TH: ดึงโมเดลที่เลือก | EN: Retrieve the selected model.
test_prediction = np.clip(selected.predict(X_test), 0, 100)  # TH: ประเมิน test หลังเลือกโมเดลแล้ว | EN: Evaluate test only after model selection.
test_result = {"model": selected_name, "split": "test", **metrics(y_test, test_prediction)}  # TH: สร้างผลทดสอบสุดท้าย | EN: Build the final test result.
results = pd.concat([validation, pd.DataFrame([test_result])], ignore_index=True)  # TH: รวมผล validation และ test | EN: Combine validation and test metrics.
results.to_csv(artifacts / "health_model_metrics.csv", index=False)  # TH: บันทึกตารางตัวชี้วัด | EN: Save the metric table.
output = test[[column for column in ["sample_id", "site_id", "spatial_block", "data_status"] if column in test.columns]].copy()  # TH: สร้างตารางผลพยากรณ์พร้อมรหัส | EN: Build a traceable prediction table.
output[f"{TARGET}_actual"] = y_test.to_numpy()  # TH: เพิ่มค่าจริงลงตาราง | EN: Add observed target values.
output[f"{TARGET}_prediction"] = test_prediction  # TH: เพิ่มค่าพยากรณ์ลงตาราง | EN: Add predicted target values.
output.to_csv(artifacts / "health_test_predictions.csv", index=False)  # TH: บันทึกผลพยากรณ์ test | EN: Save test predictions.
joblib.dump(selected, artifacts / "selected_health_model.joblib")  # TH: บันทึกโมเดลที่เลือกด้วยชื่อกลาง | EN: Save the selected model under a stable name.
selected_meta = {"selected_model": selected_name, "target": TARGET, "features": FEATURES, "selection_metric": "validation_RMSE", "test_metrics": test_result, "health_mode": META["health_mode"], "seed": SEED}  # TH: สร้างข้อมูลกำกับโมเดล | EN: Build model metadata.
(artifacts / "selected_health_model.json").write_text(json.dumps(selected_meta, ensure_ascii=False, indent=2), encoding="utf-8")  # TH: บันทึกข้อมูลกำกับโมเดล | EN: Save model metadata.
print(results)  # TH: แสดงผลทั้งหมด | EN: Display all results.
